# Linear Transformations — Exercises

8 exercises covering the full Linear Transformations curriculum: from kernel/image computation through Jacobian derivation and LoRA rank analysis.

| Format | Description |
| --- | --- |
| **Problem** | Markdown cell with task description |
| **Your Solution** | Code scaffold — replace `# YOUR CODE HERE` |
| **Solution** | Reference solution with PASS/FAIL checks |

### Difficulty Levels

| Level | Exercises | Focus |
| --- | --- | --- |
| ★ | 1–3 | Core mechanics |
| ★★ | 4–6 | Deeper theory |
| ★★★ | 7–8 | AI / ML applications |

### Topic Map

| Topic | Exercise |
| --- | --- |
| Kernel, image, rank-nullity | 1 |
| Matrix of T in non-standard basis | 2 |
| Change of basis | 3 |
| Projection operator construction | 4 |
| Jacobian of softmax | 5 |
| Affine map composition | 6 |
| LoRA rank analysis | 7 |
| Linear probing | 8 |


In [ ]:
import numpy as np
import numpy.linalg as la

try:
    import matplotlib.pyplot as plt
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['figure.figsize'] = [10, 5]
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-6):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print(f'  expected: {np.round(np.array(expected), 6)}')
        print(f'  got     : {np.round(np.array(got), 6)}')
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

def kernel_basis(A, tol=1e-8):
    """Compute a basis for ker(A) via SVD."""
    _, s, Vt = la.svd(A)
    r = np.sum(s > tol)
    return Vt[r:].T  # columns span the null space

def image_basis(A, tol=1e-8):
    """Compute a basis for im(A) via SVD."""
    U, s, _ = la.svd(A, full_matrices=True)
    r = np.sum(s > tol)
    return U[:, :r]  # columns span the image

print('Setup complete.')


---

## Exercise 1 ★ — Kernel, Image, and Rank-Nullity

**Goal:** Compute kernel and image for explicit matrices and verify rank-nullity.

**(a)** For $A = \begin{pmatrix}1 & 2 & 3 \\ 4 & 5 & 6 \\ 7 & 8 & 9\end{pmatrix}$: find a basis for $\ker(A)$, a basis for $\operatorname{im}(A)$, verify $\operatorname{rank}(A) + \operatorname{nullity}(A) = 3$.

**(b)** For the horizontal projection $B = \begin{pmatrix}1&0\\0&0\end{pmatrix}$: state (without computation) what $\ker(B)$ and $\operatorname{im}(B)$ are. Verify.

**(c)** For a random $5 \times 8$ matrix of rank 3: verify rank-nullity. How many free variables does $C\mathbf{x} = \mathbf{0}$ have?

In [ ]:
# Exercise 1: Your Solution

# (a) 3x3 rank-deficient matrix
A = np.array([[1., 2., 3.],
              [4., 5., 6.],
              [7., 8., 9.]])

def compute_rank_nullity(A):
    """Return (rank, nullity, kernel_basis, image_basis)."""
    # YOUR CODE HERE
    rank = None
    nullity = None
    ker = None
    im = None
    return rank, nullity, ker, im

rank_A, nullity_A, ker_A, im_A = compute_rank_nullity(A)
print(f'(a) rank={rank_A}, nullity={nullity_A}')
print(f'    Rank-nullity check: {rank_A} + {nullity_A} = {rank_A + nullity_A if rank_A and nullity_A else None}')

# (b) Projection matrix
B = np.array([[1., 0.], [0., 0.]])
ker_B = None   # YOUR CODE HERE: basis for ker(B)
im_B = None    # YOUR CODE HERE: basis for im(B)
print(f'(b) ker(B) basis: {ker_B}, im(B) basis: {im_B}')

# (c) Random rank-3 matrix
np.random.seed(5)
U = la.qr(np.random.randn(5, 3))[0]
V = la.qr(np.random.randn(8, 3))[0]
C = U @ np.diag([2., 1., 0.5]) @ V.T
rank_C, nullity_C, ker_C, im_C = compute_rank_nullity(C)
print(f'(c) C is 5x8 with rank={rank_C}')
print(f'    Nullity={nullity_C}, free variables in Cx=0: {nullity_C}')


In [ ]:
# Exercise 1: Solution

import numpy as np
import numpy.linalg as la

def compute_rank_nullity(A, tol=1e-8):
    """Return (rank, nullity, kernel_basis, image_basis)."""
    U, s, Vt = la.svd(A, full_matrices=True)
    rank = int(np.sum(s > tol))
    nullity = A.shape[1] - rank
    ker = Vt[rank:].T  # null space
    im = U[:, :rank]   # column space
    return rank, nullity, ker, im

A = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]])
B = np.array([[1., 0.], [0., 0.]])
np.random.seed(5)
U0 = la.qr(np.random.randn(5, 3))[0]
V0 = la.qr(np.random.randn(8, 3))[0]
C = U0 @ np.diag([2., 1., 0.5]) @ V0.T

header('Exercise 1: Kernel, Image, Rank-Nullity')

# (a)
rank_A, nullity_A, ker_A, im_A = compute_rank_nullity(A)
check_true('(a) rank(A) = 2 (rows are linearly dependent)', rank_A == 2)
check_true('(a) nullity(A) = 1 (one free variable)', nullity_A == 1)
check_true('(a) rank + nullity = 3 (n_cols)', rank_A + nullity_A == A.shape[1])
check_close('(a) A @ ker = 0', A @ ker_A, np.zeros((3, nullity_A)))

# (b)
rank_B, nullity_B, ker_B, im_B = compute_rank_nullity(B)
check_true('(b) ker(B) is y-axis: dim 1', nullity_B == 1)
check_true('(b) im(B) is x-axis: dim 1', rank_B == 1)
check_close('(b) kernel vector is e_2 = (0,1)', abs(ker_B[:,0]), np.array([0., 1.]))
check_close('(b) image vector is e_1 = (1,0)', abs(im_B[:,0]), np.array([1., 0.]))

# (c)
rank_C, nullity_C, ker_C, im_C = compute_rank_nullity(C)
check_true('(c) rank(C) = 3', rank_C == 3)
check_true('(c) nullity(C) = 5 (8-3)', nullity_C == 5)
check_true('(c) rank + nullity = 8', rank_C + nullity_C == C.shape[1])
check_close('(c) C @ ker_C = 0', C @ ker_C, np.zeros((5, 5)))

print('\nTakeaway: Rank-nullity tells us exactly how much of the input')
print('is preserved (rank) vs collapsed to zero (nullity) by any linear map.')


---

## Exercise 2 ★ — Matrix of T in Non-Standard Basis

**Goal:** Construct the matrix of a linear map in a specified non-standard basis.

Let $T: \mathbb{R}^2 \to \mathbb{R}^2$ be reflection across the line $y = x$ (i.e., $T(x, y) = (y, x)$).

Basis $\mathcal{B} = \{\mathbf{b}_1, \mathbf{b}_2\}$ where $\mathbf{b}_1 = (1,1)/\sqrt{2}$ and $\mathbf{b}_2 = (1,-1)/\sqrt{2}$.

**(a)** Write the standard matrix $A$ of $T$.

**(b)** Compute the change-of-basis matrix $P$ from $\mathcal{B}$ to the standard basis.

**(c)** Compute $[T]_{\mathcal{B}} = P^{-1}AP$. What is geometrically notable about this result?

**(d)** Verify: in basis $\mathcal{B}$, what do $T(\mathbf{b}_1)$ and $T(\mathbf{b}_2)$ equal?

In [ ]:
# Exercise 2: Your Solution

import numpy as np
import numpy.linalg as la

# (a) Standard matrix of reflection across y=x
A = None  # YOUR CODE HERE

# (b) Change-of-basis matrix P
b1 = np.array([1., 1.]) / np.sqrt(2)
b2 = np.array([1., -1.]) / np.sqrt(2)
P = None  # YOUR CODE HERE: columns are b1, b2

# (c) Matrix of T in basis B
T_in_B = None  # YOUR CODE HERE: P_inv @ A @ P
print('T in standard basis A:')
print(A)
print('T in basis B =')
print(T_in_B)

# (d) What are T(b1) and T(b2)?
Tb1 = None  # YOUR CODE HERE: T applied to b1
Tb2 = None  # YOUR CODE HERE: T applied to b2
print(f'T(b1) = {Tb1}')
print(f'T(b2) = {Tb2}')


In [ ]:
# Exercise 2: Solution

import numpy as np
import numpy.linalg as la

# (a) Reflection across y=x: (x,y) -> (y,x)
A = np.array([[0., 1.], [1., 0.]])

# (b) Change-of-basis matrix
b1 = np.array([1., 1.]) / np.sqrt(2)
b2 = np.array([1., -1.]) / np.sqrt(2)
P = np.column_stack([b1, b2])

# (c) Matrix of T in basis B
T_in_B = la.inv(P) @ A @ P

# (d) T(b1) = b1 (b1 lies ON the line y=x, reflection fixes it)
#     T(b2) = -b2 (b2 is perpendicular to y=x, gets negated)
Tb1 = A @ b1
Tb2 = A @ b2

header = lambda t: (print('\n' + '='*len(t)), print(t), print('='*len(t)))
header('Exercise 2: Matrix in Non-Standard Basis')

print(f'Standard matrix A:\n{A}')
print(f'\nP (change-of-basis):\n{np.round(P, 4)}')
print(f'\n[T]_B = P^{{-1}}AP:\n{np.round(T_in_B, 6)}')
print()

check_close = lambda n, g, e: print(f"{'PASS' if np.allclose(g, e, atol=1e-8) else 'FAIL'} — {n}")
check_close('[T]_B is diagonal with (1, -1)', T_in_B, np.diag([1., -1.]))
check_close('T(b1) = b1 (b1 fixed by reflection)', Tb1, b1)
check_close('T(b2) = -b2 (b2 negated by reflection)', Tb2, -b2)

print('\nGeometric insight:')
print('In the basis {b1, b2}, reflection across y=x is DIAGONAL: diag(1,-1).')
print('b1=(1,1)/sqrt(2) is along y=x (eigenvalue +1, fixed).')
print('b2=(1,-1)/sqrt(2) is perp to y=x (eigenvalue -1, flipped).')
print('\nTakeaway: The right basis turns a reflection into simple scaling!')


---

## Exercise 3 ★ — Rank-Nullity and Linear Systems

**Goal:** Use rank-nullity to understand solution structures.

**(a)** $A \in \mathbb{R}^{4 \times 6}$ has rank 3. What is the nullity? How many free variables are there in $A\mathbf{x} = \mathbf{0}$?

**(b)** $B \in \mathbb{R}^{6 \times 4}$ has rank 4. Is $B\mathbf{x} = \mathbf{b}$ always solvable? What is nullity of $B$?

**(c)** Prove: If $T: V \to V$ is a linear map on finite-dimensional $V$, then $T$ is injective iff $T$ is surjective. Demonstrate numerically.

In [ ]:
# Exercise 3: Your Solution

import numpy as np
import numpy.linalg as la

# (a) 4x6 rank-3 matrix
nullity_a = None  # YOUR CODE HERE: nullity of 4x6 rank-3 matrix
free_vars_a = None  # YOUR CODE HERE
print(f'(a) Nullity = {nullity_a}, free variables = {free_vars_a}')

# (b) 6x4 rank-4 matrix
nullity_b = None  # YOUR CODE HERE
always_solvable = None  # YOUR CODE HERE: True or False
print(f'(b) Nullity = {nullity_b}, always solvable = {always_solvable}')

# (c) Numerical demonstration: injective <=> surjective for T: V -> V
np.random.seed(1)
n = 4
T_matrix = np.random.randn(n, n)  # generically full rank
is_injective = None  # YOUR CODE HERE: based on ker
is_surjective = None  # YOUR CODE HERE: based on im
print(f'(c) Square matrix: injective={is_injective}, surjective={is_surjective}')

# Now rank-deficient square
T_deficient = np.outer(np.random.randn(n), np.random.randn(n))  # rank 1
is_inj2 = None  # YOUR CODE HERE
is_sur2 = None  # YOUR CODE HERE
print(f'(c) Rank-1 square: injective={is_inj2}, surjective={is_sur2}')


In [ ]:
# Exercise 3: Solution

import numpy as np
import numpy.linalg as la

np.random.seed(1)

def diagnose(A, label):
    m, n = A.shape
    r = la.matrix_rank(A)
    nullity = n - r
    injective = (nullity == 0)
    surjective = (r == m)
    print(f'{label}: shape {m}x{n}, rank {r}, nullity {nullity}')
    print(f'  injective={injective}, surjective={surjective}')
    return r, nullity, injective, surjective

header = lambda t: (print('\n'+'='*len(t)), print(t), print('='*len(t)))
header('Exercise 3: Rank-Nullity and Linear Systems')

check_true = lambda n, c: print(f"{'PASS' if c else 'FAIL'} — {n}")

# (a) 4x6, rank 3
# Build a 4x6 rank-3 matrix
A = la.qr(np.random.randn(4,3))[0] @ np.diag([2,1,0.5]) @ la.qr(np.random.randn(6,3))[0].T
r_a, null_a, inj_a, sur_a = diagnose(A, '(a) 4x6 rank-3')
check_true('(a) nullity = 3 (= 6-3)', null_a == 3)
check_true('(a) not injective (nullity > 0)', not inj_a)
check_true('(a) not surjective (rank < 4)', not sur_a)
print(f'  Free variables in Ax=0: {null_a}')

# (b) 6x4, rank 4
B = np.random.randn(6, 4)
r_b, null_b, inj_b, sur_b = diagnose(B, '(b) 6x4 rank-4')
check_true('(b) nullity = 0 (injective)', null_b == 0)
check_true('(b) NOT surjective (rank=4 < 6)', not sur_b)
print('  Bx=b NOT always solvable: b must be in col(B) (a 4-dim subspace of R^6)')

# (c) Square matrices
T_full = np.random.randn(4, 4)  # full rank
T_def = np.outer(np.random.randn(4), np.random.randn(4))  # rank 1

r_f, null_f, inj_f, sur_f = diagnose(T_full, '(c) 4x4 full rank')
check_true('(c) full rank: injective iff surjective', inj_f == sur_f)

r_d, null_d, inj_d, sur_d = diagnose(T_def, '(c) 4x4 rank-1')
check_true('(c) rank-1: neither injective nor surjective', not inj_d and not sur_d)

print('\nProof sketch: For T:V->V, rank-nullity gives rank + nullity = n.')
print('nullity=0 <=> rank=n <=> im(T)=V. So injective <=> surjective.')
print('\nTakeaway: In finite dims, for ENDOMORPHISMS only, inj <=> surj <=> bij!')


---

## Exercise 4 ★★ — Projection Operator Construction

**Goal:** Build an orthogonal projection onto a given subspace and verify all properties.

Let $S = \operatorname{span}\{(1, 2, 0)^\top, (0, 1, 1)^\top\} \subseteq \mathbb{R}^3$.

**(a)** Orthonormalize the spanning set using Gram-Schmidt.

**(b)** Construct the projection matrix $P = QQ^\top$ (where $Q$ has orthonormal basis as columns).

**(c)** Verify: $P^2 = P$, $P = P^\top$, and $\operatorname{rank}(P) = \operatorname{tr}(P) = 2$.

**(d)** Compute $(I-P)$ and verify it is also a projection. Show $Pv + (I-P)v = v$ for any $v$.

In [ ]:
# Exercise 4: Your Solution

import numpy as np
import numpy.linalg as la

v1 = np.array([1., 2., 0.])
v2 = np.array([0., 1., 1.])

def gram_schmidt(vecs):
    """Orthonormalize list of vectors."""
    # YOUR CODE HERE
    pass

# (a) Orthonormalize
Q_cols = gram_schmidt([v1, v2])
Q = None  # YOUR CODE HERE: assemble into matrix

# (b) Projection matrix
P = None  # YOUR CODE HERE: Q @ Q.T

# (c) Verify properties
print('P ='); print(P)
print('P^2 == P:', np.allclose(P @ P, P) if P is not None else None)
print('P == P^T:', np.allclose(P, P.T) if P is not None else None)

# (d) Complementary projection
I = np.eye(3)
Q_comp = None  # YOUR CODE HERE: I - P
v_test = np.array([1., 2., 3.])
print(f'P @ v + (I-P) @ v = v: {np.allclose((P @ v_test + Q_comp @ v_test), v_test) if P is not None else None}')


In [ ]:
# Exercise 4: Solution

import numpy as np
import numpy.linalg as la

v1 = np.array([1., 2., 0.])
v2 = np.array([0., 1., 1.])

def gram_schmidt(vecs):
    """Orthonormalize a list of vectors."""
    result = []
    for v in vecs:
        q = v.copy().astype(float)
        for u in result:
            q -= (q @ u) * u  # subtract projection onto previous
        norm = la.norm(q)
        if norm > 1e-10:
            result.append(q / norm)
    return result

# (a) Orthonormalize
q_cols = gram_schmidt([v1, v2])
Q = np.column_stack(q_cols)

# (b) Projection matrix
P = Q @ Q.T

# (c) Complementary
I = np.eye(3)
P_comp = I - P

check_close = lambda n, g, e: print(f"{'PASS' if np.allclose(g, e, atol=1e-8) else 'FAIL'} — {n}")
check_true = lambda n, c: print(f"{'PASS' if c else 'FAIL'} — {n}")

header = lambda t: (print('\n'+'='*len(t)), print(t), print('='*len(t)))
header('Exercise 4: Projection Operator')

print(f'Q (orthonormal basis for S):\n{np.round(Q, 4)}')
print(f'P = Q Q^T:\n{np.round(P, 4)}')

check_close('(c) P^2 = P (idempotent)', P @ P, P)
check_close('(c) P = P^T (symmetric)', P, P.T)
check_true('(c) rank(P) = 2', la.matrix_rank(P) == 2)
check_close('(c) trace(P) = rank(P) = 2', np.trace(P), 2.0)

check_close('(d) (I-P)^2 = I-P', P_comp @ P_comp, P_comp)
check_close('(d) P(I-P) = 0', P @ P_comp, np.zeros((3,3)))

v_test = np.array([1., 2., 3.])
check_close('(d) Pv + (I-P)v = v', P @ v_test + P_comp @ v_test, v_test)
check_close('(d) Pv perp to (I-P)v', float((P @ v_test) @ (P_comp @ v_test)), 0.0)

print('\nTakeaway: Every orthogonal projection P satisfies P^2=P, P=P^T.')
print('V = im(P) ⊕ ker(P), and these two subspaces are orthogonal.')
print('rank(P) = tr(P) = number of eigenvalues equal to 1.')


---

## Exercise 5 ★★ — Jacobian of Softmax

**Goal:** Derive and verify the Jacobian of the softmax function.

For $\mathbf{s} = \operatorname{softmax}(\mathbf{z})$ where $s_i = e^{z_i}/\sum_k e^{z_k}$:

**(a)** Compute $\partial s_i/\partial z_j$ for the cases $i=j$ and $i \neq j$.

**(b)** Show that the full Jacobian is $J = \operatorname{diag}(\mathbf{s}) - \mathbf{s}\mathbf{s}^\top$.

**(c)** Verify that $J\mathbf{1} = \mathbf{0}$ (Jacobian kills constant vectors).

**(d)** Show that $\operatorname{rank}(J) \leq n-1$.

In [ ]:
# Exercise 5: Your Solution

import numpy as np

z = np.array([1.0, 2.0, 0.5, -0.5])

def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

def jacobian_softmax(z):
    """Compute Jacobian of softmax at z."""
    # YOUR CODE HERE
    # Hint: J = diag(s) - outer(s, s) where s = softmax(z)
    pass

J = jacobian_softmax(z)
s = softmax(z)

print('s =', np.round(s, 4) if s is not None else None)
print('J ='); print(np.round(J, 4) if J is not None else None)
print('J @ 1 =', np.round(J @ np.ones(4), 6) if J is not None else None)
print('rank(J) =', np.linalg.matrix_rank(J) if J is not None else None)


In [ ]:
# Exercise 5: Solution

import numpy as np
import numpy.linalg as la

z = np.array([1.0, 2.0, 0.5, -0.5])

def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

def jacobian_softmax(z):
    """J_ij = ds_i/dz_j = s_i*(delta_ij - s_j) = diag(s) - s s^T."""
    s = softmax(z)
    return np.diag(s) - np.outer(s, s)

def jacobian_fd(f, z, eps=1e-5):
    """Finite-difference Jacobian."""
    n = len(z)
    J = np.zeros((n, n))
    for j in range(n):
        zp, zm = z.copy(), z.copy()
        zp[j] += eps; zm[j] -= eps
        J[:, j] = (f(zp) - f(zm)) / (2 * eps)
    return J

s = softmax(z)
J = jacobian_softmax(z)
J_fd = jacobian_fd(softmax, z)

check_close = lambda n, g, e: print(f"{'PASS' if np.allclose(g, e, atol=1e-5) else 'FAIL'} — {n}")
check_true = lambda n, c: print(f"{'PASS' if c else 'FAIL'} — {n}")

header = lambda t: (print('\n'+'='*len(t)), print(t), print('='*len(t)))
header('Exercise 5: Jacobian of Softmax')

print(f's = {np.round(s, 4)}')
print(f'J = diag(s) - ss^T:\n{np.round(J, 4)}')
print()

check_close('(b) J matches finite-difference Jacobian', J, J_fd)
check_close('(c) J @ 1 = 0 (kills constants)', J @ np.ones(4), np.zeros(4))

rank_J = la.matrix_rank(J)
check_true(f'(d) rank(J) <= n-1 = {len(z)-1}', rank_J <= len(z) - 1)
print(f'  actual rank(J) = {rank_J}')

# Why rank is n-1:
# J has null vector 1 (J@1=0), so rank <= n-1.
# J is PSD (diag(s) - ss^T >= 0), with exactly one zero eigenvalue.
evals = np.sort(la.eigvalsh(J))[::-1]
print(f'  eigenvalues of J: {np.round(evals, 4)}')

print('\nTakeaway: Softmax Jacobian has rank n-1 because softmax outputs sum to 1,')
print('so any constant perturbation to z leaves softmax unchanged — J kills constants.')
print('This is why the cross-entropy gradient is (s - y), not J @ (s - y).')


---

## Exercise 6 ★★ — Affine Map Composition via Homogeneous Coordinates

**Goal:** Compose affine transformations using augmented matrices.

In $\mathbb{R}^2$, define:
- $f_1$: rotate by $45°$ then translate by $(1, 0)$
- $f_2$: scale by factor $2$ in both dimensions then translate by $(0, 1)$

**(a)** Write augmented $3 \times 3$ matrices $\tilde{M}_1$ and $\tilde{M}_2$.

**(b)** Compute $\tilde{M}_2 \tilde{M}_1$ (apply $f_1$ then $f_2$). What are the effective rotation, scale, and translation?

**(c)** Apply the composition to the point $(1, 0)$. Verify by applying $f_1$ then $f_2$ directly.

In [ ]:
# Exercise 6: Your Solution

import numpy as np

def make_affine(A, b):
    """Build 3x3 homogeneous matrix for affine map x -> Ax + b in R^2."""
    # YOUR CODE HERE
    pass

def apply_affine(M, x):
    """Apply homogeneous matrix M to 2D point x."""
    # YOUR CODE HERE
    pass

# f1: rotate 45 degrees + translate (1,0)
theta = np.pi / 4
R45 = np.array([[np.cos(theta), -np.sin(theta)],
                [np.sin(theta),  np.cos(theta)]])
M1 = make_affine(R45, np.array([1., 0.]))

# f2: scale by 2 + translate (0,1)
M2 = make_affine(2 * np.eye(2), np.array([0., 1.]))

print('M1 =', M1)
print('M2 =', M2)

# (b) Composed matrix
M_comp = None  # YOUR CODE HERE: M2 @ M1
print('M_comp = M2 @ M1 =', M_comp)

# (c) Apply to x = (1, 0)
x = np.array([1., 0.])
y_composed = apply_affine(M_comp, x)
print(f'Composed result: {y_composed}')


In [ ]:
# Exercise 6: Solution

import numpy as np

def make_affine(A, b):
    m, n = A.shape
    M = np.eye(m + 1)
    M[:m, :n] = A
    M[:m, n] = b
    return M

def apply_affine(M, x):
    x_hom = np.append(x, 1.0)
    return (M @ x_hom)[:-1]

theta = np.pi / 4
R45 = np.array([[np.cos(theta), -np.sin(theta)],
                [np.sin(theta),  np.cos(theta)]])
M1 = make_affine(R45, np.array([1., 0.]))
M2 = make_affine(2.0 * np.eye(2), np.array([0., 1.]))
M_comp = M2 @ M1

x = np.array([1., 0.])

# Sequential: f1 then f2
y1 = apply_affine(M1, x)    # after f1
y2 = apply_affine(M2, y1)   # after f2
y_comp = apply_affine(M_comp, x)

check_close = lambda n, g, e: print(f"{'PASS' if np.allclose(g, e, atol=1e-8) else 'FAIL'} — {n}")

header = lambda t: (print('\n'+'='*len(t)), print(t), print('='*len(t)))
header('Exercise 6: Affine Map Composition')

print(f'M1:\n{np.round(M1, 4)}')
print(f'M2:\n{np.round(M2, 4)}')
print(f'M_comp = M2 @ M1:\n{np.round(M_comp, 4)}')
print()
print(f'f1({x}) = {np.round(y1, 4)}')
print(f'f2(f1({x})) = {np.round(y2, 4)}')
print(f'Composed({x}) = {np.round(y_comp, 4)}')
print()

check_close('(c) Composed = sequential application', y_comp, y2)

# Read off effective transform from M_comp
A_eff = M_comp[:2, :2]
b_eff = M_comp[:2, 2]
print(f'\nEffective linear part: 2*R45 =\n{np.round(A_eff, 4)}')
print(f'Effective translation: {np.round(b_eff, 4)}')

print('\nTakeaway: Homogeneous coordinates make affine composition = matrix multiplication.')
print('Precomputing M_comp saves cost when applying to many points.')


---

## Exercise 7 ★★★ — LoRA Rank Analysis

**Goal:** Analyze the geometry of low-rank weight updates in LoRA.

$W_0 \in \mathbb{R}^{512 \times 512}$ (frozen). LoRA update: $\Delta W = BA$, $B \in \mathbb{R}^{512 \times r}$, $A \in \mathbb{R}^{r \times 512}$.

**(a)** What is $\operatorname{rank}(\Delta W)$? What is nullity?

**(b)** Numerically verify $\operatorname{rank}(BA) \leq r$ for $r = 4$.

**(c)** For a fixed $\mathbf{x}$, show $\Delta W\mathbf{x} = BA\mathbf{x} \in \operatorname{col}(B)$.

**(d)** Compute singular values of $\Delta W$. How many are nonzero?

**(e)** Compare parameter counts for $r = 4, 8, 16, 32$.

In [ ]:
# Exercise 7: Your Solution

import numpy as np
import numpy.linalg as la

np.random.seed(42)
d, k, r = 512, 512, 4

# LoRA factors
B_lora = np.random.randn(d, r) * 0.01
A_lora = np.random.randn(r, k) * 0.01
delta_W = B_lora @ A_lora

# (a) What is rank(delta_W)?
rank_dW = None  # YOUR CODE HERE
nullity_dW = None  # YOUR CODE HERE
print(f'(a) rank(delta_W) = {rank_dW}, nullity = {nullity_dW}')

# (b) Verify rank <= r
print(f'(b) rank(BA) = {rank_dW}, r = {r}, rank <= r: {rank_dW <= r if rank_dW else None}')

# (c) Show delta_W @ x is in col(B)
x = np.random.randn(k)
output = delta_W @ x  # = B @ (A @ x) = B @ intermediate
# YOUR CODE HERE: verify output is in col(B)
in_col_B = None
print(f'(c) delta_W @ x in col(B): {in_col_B}')

# (d) Singular values
svs = None  # YOUR CODE HERE
nonzero_svs = None  # YOUR CODE HERE
print(f'(d) Nonzero singular values: {nonzero_svs}')

# (e) Parameter counts
for r_val in [4, 8, 16, 32]:
    params_lora = d * r_val + r_val * k
    params_full = d * k
    print(f'r={r_val:2d}: LoRA params={params_lora:,}, savings={params_full//params_lora}x')


In [ ]:
# Exercise 7: Solution

import numpy as np
import numpy.linalg as la

np.random.seed(42)
d, k, r = 512, 512, 4

B_lora = np.random.randn(d, r) * 0.01
A_lora = np.random.randn(r, k) * 0.01
delta_W = B_lora @ A_lora

check_close = lambda n, g, e, tol=1e-6: print(f"{'PASS' if np.allclose(g, e, atol=tol) else 'FAIL'} — {n}")
check_true = lambda n, c: print(f"{'PASS' if c else 'FAIL'} — {n}")

header = lambda t: (print('\n'+'='*len(t)), print(t), print('='*len(t)))
header('Exercise 7: LoRA Rank Analysis')

# (a) & (b)
rank_dW = la.matrix_rank(delta_W, tol=1e-6)
nullity_dW = k - rank_dW
print(f'(a) rank(delta_W) = {rank_dW} (expected <= r={r})')
print(f'    nullity = {nullity_dW} (= {k}-{rank_dW})')
print(f'    {100*nullity_dW/k:.1f}% of input space is unaffected by the update')
check_true('(b) rank(BA) <= r', rank_dW <= r)

# (c) delta_W @ x = B @ (A @ x), which is B @ (something in R^r) = col(B)
x = np.random.randn(k)
Ax = A_lora @ x   # in R^r
output = delta_W @ x  # = B @ Ax
output_direct = B_lora @ Ax
check_close('(c) delta_W @ x = B @ (A @ x)', output, output_direct)

# Verify output is in col(B): project onto col(B) should be identity
Q_B, _ = la.qr(B_lora)  # orthonormal basis for col(B)
Q_B = Q_B[:, :r]
proj_output = Q_B @ (Q_B.T @ output)  # project onto col(B)
check_close('(c) output is in col(B) (projection = self)', output, proj_output, tol=1e-6)

# (d) Singular values
svs = la.svd(delta_W, compute_uv=False)
nonzero = np.sum(svs > 1e-6)
print(f'\n(d) Top-5 singular values: {np.round(svs[:5], 6)}')
print(f'    Singular values [r] through [r+3]: {np.round(svs[r:r+4], 10)}')
check_true(f'(d) Exactly {r} nonzero singular values', nonzero == r)

# (e) Parameter efficiency
print('\n(e) Parameter efficiency:')
params_full = d * k
for r_val in [4, 8, 16, 32]:
    params_lora = d * r_val + r_val * k
    print(f'  r={r_val:2d}: LoRA = {params_lora:,}, full = {params_full:,}, '
          f'savings = {params_full/params_lora:.0f}x')

print('\nTakeaway: LoRA uses rank-nullity to justify parameter efficiency.')
print('If task-relevant updates have rank r << min(d,k),')
print('we only need r(d+k) parameters instead of dk.')


---

## Exercise 8 ★★★ — Linear Probing and Linear Representation Hypothesis

**Goal:** Test whether concepts are linearly represented in embedding space.

**(a)** Generate synthetic embeddings: 2 binary features along orthogonal directions in $\mathbb{R}^{50}$, with noise.

**(b)** Train a linear probe for feature 0 and measure accuracy.

**(c)** Show that PC1 (from PCA) aligns with the true feature direction.

**(d)** Demonstrate superposition: 2 features in $\mathbb{R}^3$ (fewer dims than features).

**(e)** Compute mutual coherence of feature directions. How does it relate to probe accuracy?

In [ ]:
# Exercise 8: Your Solution

import numpy as np
import numpy.linalg as la

np.random.seed(42)
n, d, n_features = 300, 50, 2

# (a) Generate embeddings
# YOUR CODE HERE: create feature directions, labels, signal, noise, embeddings
feature_dirs = None
labels = None
embeddings = None

# (b) Linear probe for feature 0
probe_acc = None  # YOUR CODE HERE
print(f'(b) Linear probe accuracy: {probe_acc}')

# (c) PCA alignment
pc1 = None  # YOUR CODE HERE: top PC of embeddings
alignment = None  # YOUR CODE HERE: |PC1 . feature_dir_0|
print(f'(c) Alignment of PC1 with feature dir: {alignment}')

# (d) Superposition: 2 features in 3 dims
d_small = 3
# YOUR CODE HERE: create superposed embeddings
super_acc = None
print(f'(d) Probe accuracy in superposition (d=3, 2 features): {super_acc}')

# (e) Mutual coherence
coherence = None  # YOUR CODE HERE: max |d1 . d2| for distinct directions
print(f'(e) Mutual coherence: {coherence}')


In [ ]:
# Exercise 8: Solution

import numpy as np
import numpy.linalg as la

np.random.seed(42)
n, d, n_features = 300, 50, 2

# (a) Generate embeddings with linear features
# Orthogonal feature directions in R^d
D_raw = np.random.randn(d, n_features)
feature_dirs, _ = la.qr(D_raw)  # orthonormalize
feature_dirs = feature_dirs[:, :n_features]

labels = np.random.randint(0, 2, (n, n_features)).astype(float)
signal = (2 * labels - 1) @ feature_dirs.T  # ±1 along each direction
noise = 3.0 * np.random.randn(n, d)
embeddings = signal + noise

check_close = lambda nm, g, e, t=1e-6: print(f"{'PASS' if np.allclose(g,e,atol=t) else 'FAIL'} — {nm}")
check_true = lambda nm, c: print(f"{'PASS' if c else 'FAIL'} — {nm}")

header = lambda t: (print('\n'+'='*len(t)), print(t), print('='*len(t)))
header('Exercise 8: Linear Probing')

# (b) Linear probe for feature 0
n_train = 200
X_tr, X_te = embeddings[:n_train], embeddings[n_train:]
y_tr, y_te = labels[:n_train, 0], labels[n_train:, 0]

X_tr_aug = np.column_stack([X_tr, np.ones(n_train)])
X_te_aug = np.column_stack([X_te, np.ones(len(X_te))])
w, _, _, _ = la.lstsq(X_tr_aug, y_tr, rcond=None)
y_pred = (X_te_aug @ w > 0.5).astype(float)
probe_acc = (y_pred == y_te).mean()
print(f'(b) Linear probe accuracy: {probe_acc:.1%}')
check_true('(b) Linear probe achieves >70% accuracy', probe_acc > 0.70)

# (c) PCA alignment
X_c = embeddings - embeddings.mean(0)
_, _, Vt = la.svd(X_c, full_matrices=False)
pc1 = Vt[0]
alignment = abs(pc1 @ feature_dirs[:, 0])
print(f'(c) |PC1 . true_dir| = {alignment:.3f} (1.0 = perfect)')
check_true('(c) PC1 aligns with feature direction (|dot| > 0.5)', alignment > 0.5)

# (d) Superposition: 2 binary features in 3D space
d_small = 3
D2_raw = np.random.randn(d_small, n_features)
dirs2, _ = la.qr(np.random.randn(d_small, d_small))
dirs2 = dirs2[:, :n_features]

signal2 = (2 * labels - 1) @ dirs2.T  # (n, 3)
noise2 = 0.5 * np.random.randn(n, d_small)
emb2 = signal2 + noise2

X2_tr, X2_te = emb2[:n_train], emb2[n_train:]
X2_tr_aug = np.column_stack([X2_tr, np.ones(n_train)])
X2_te_aug = np.column_stack([X2_te, np.ones(len(X2_te))])
w2, _, _, _ = la.lstsq(X2_tr_aug, y_tr, rcond=None)
y2_pred = (X2_te_aug @ w2 > 0.5).astype(float)
super_acc = (y2_pred == y_te).mean()
print(f'(d) Superposition probe accuracy: {super_acc:.1%}')
check_true('(d) Superposition: still linearly decodable', super_acc > 0.60)

# (e) Mutual coherence of feature directions
coherence = abs(dirs2[:, 0] @ dirs2[:, 1])
print(f'(e) Mutual coherence |d1.d2| = {coherence:.3f} (0=orthog, 1=same)')
print(f'    Lower coherence -> less interference -> higher probe accuracy')

print('\nTakeaway: The linear representation hypothesis says features')
print('are encoded as directions. PCA finds them. Linear probes decode them.')
print('Superposition allows MORE features than dimensions at the cost of coherence.')


---

## What to Review After Finishing

- [ ] **Kernel and Image** — Can you compute both from a matrix using SVD?
- [ ] **Rank-Nullity** — Can you apply it to determine solution space dimensions?
- [ ] **Change of Basis** — Can you derive $[T]_{\mathcal{B}'} = P^{-1}[T]_\mathcal{B}P$?
- [ ] **Projections** — Do you understand idempotency and the orthogonal/oblique distinction?
- [ ] **Jacobian** — Can you compute Jacobians of softmax, ReLU, and linear layers?
- [ ] **Backpropagation** — Do you see why the backward pass uses $W^\top$ (the dual map)?
- [ ] **LoRA** — Can you explain why $\operatorname{rank}(BA) \leq r$ implies parameter efficiency?
- [ ] **Linear Probing** — Do you understand what it means for a feature to be linearly decodable?

## References

1. Axler, S. (2015). *Linear Algebra Done Right* — abstract linear map theory
2. Strang, G. (2016). *Introduction to Linear Algebra* — four fundamental subspaces
3. Hu, E. et al. (2021). LoRA: Low-Rank Adaptation of Large Language Models
4. Elhage, N. et al. (2021). A Mathematical Framework for Transformer Circuits
5. Park, K. et al. (2023). The Linear Representation Hypothesis
